# LettuceDetect baseline on `Ivan1008/toolace-hallucination-spans`

Implements task **(2) Evaluate existing baseline methods** from `Hallucination Detection in Tool Calling.pdf` using the encoder-based LettuceDetect model from Kovács & Recski, 2025 ([arXiv:2502.17125](https://arxiv.org/abs/2502.17125)).

**Approach (matches the LettuceDetect paper):**
- Token-classification model `KRLabsOrg/lettucedect-{base,large}-modernbert-en-v1`, ModernBERT backbone, 4096-token context window.
- Inputs are formatted as `(context, question, answer)` triples — here `context = tool output`, `question = user query`, `answer = model final response`, exactly as the task description requires.
- Tokens belonging to the answer are classified as `0=supported` / `1=hallucinated`. Spans are built by collapsing consecutive `1`-tokens.
- We evaluate **two granularities** (Tables 2 and 3 of the Lettuce paper):
  - **Example-level** — binary detection: does this record contain *any* hallucination?
  - **Span-level** — character-overlap precision/recall/F1 between predicted spans and gold spans within the answer (RAGTruth methodology).

**Dataset:** `Ivan1008/toolace-hallucination-spans` on the Hugging Face Hub. We evaluate the four published configs (`combined`, `contradiction`, `missing_tool`, `overgeneration`) on the `test` split.

**Baselines reported in the resulting table:**
1. `lettucedect-base-modernbert-en-v1` (≈150M params)
2. `lettucedect-large-modernbert-en-v1` (≈396M params)
3. A *lexical* sanity baseline (a token in the answer is hallucinated iff it does not appear in the context) — this is *not* from the Lettuce paper, but is included as the trivial floor every encoder must beat.

Both Lettuce checkpoints were trained on RAGTruth (news/QA/data-to-text), so this is a strict **zero-shot domain-transfer** evaluation onto tool-calling dialogues.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm

from lettucedetect.models.inference import HallucinationDetector

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"torch: {torch.__version__}  device: {DEVICE}")

DATASET_REPO = "Ivan1008/toolace-hallucination-spans"
CONFIGS = ["combined", "contradiction", "missing_tool", "overgeneration"]
SPLIT = "test"
MODEL_IDS = [
    "KRLabsOrg/lettucedect-base-modernbert-en-v1",
    "KRLabsOrg/lettucedect-large-modernbert-en-v1",
]
MAX_RECORDS_PER_CONFIG = None  # set e.g. 50 for a quick smoke test
OUTPUT_DIR = Path("results/lettucedetect_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the dataset

Each record is RAGTruth-shaped:
- `query` — user query → used as `question` for LettuceDetect
- `context` — serialized tool output(s) → used as `context`
- `output` — assistant final answer → used as `answer` (predictions are over its character offsets)
- `hallucination_labels` — list of `{start, end, text, label}` over `output[start:end]`
- `meta.corruption_type` — one of `clean`, `contradiction`, `missing_tool`, `overgeneration`

In [ ]:
datasets_by_config: dict[str, list[dict]] = {}
for cfg in CONFIGS:
    ds = load_dataset(DATASET_REPO, cfg, split=SPLIT)
    records = [dict(row) for row in ds]
    if MAX_RECORDS_PER_CONFIG:
        records = records[:MAX_RECORDS_PER_CONFIG]
    datasets_by_config[cfg] = records

rows = []
for cfg, records in datasets_by_config.items():
    types = Counter(r["meta"]["corruption_type"] for r in records)
    rows.append({
        "config": cfg,
        "n": len(records),
        "clean": types.get("clean", 0),
        "contradiction": types.get("contradiction", 0),
        "missing_tool": types.get("missing_tool", 0),
        "overgeneration": types.get("overgeneration", 0),
    })
pd.DataFrame(rows).set_index("config")

## 3. Metrics

**Example-level** (Lettuce Table 2): a record is *positive* iff it has any hallucination. Predicted positive iff the model emits at least one span. Compute precision/recall/F1 over records.

**Span-level** (Lettuce Table 3): RAGTruth-style character-overlap. For each record, the *gold* mask is `True` for character indices covered by any `hallucination_labels` span; the *pred* mask is `True` for character indices covered by any predicted span. Then aggregate TP/FP/FN over characters across the whole split (micro-average).

Both are exactly the two reporting protocols used in the LettuceDetect paper.

In [ ]:
def char_mask(text: str, spans: list[dict]) -> list[bool]:
    mask = [False] * len(text)
    for s in spans:
        start = max(0, int(s.get("start", 0)))
        end = min(len(text), int(s.get("end", 0)))
        for i in range(start, end):
            mask[i] = True
    return mask


def prf1(tp: int, fp: int, fn: int) -> dict:
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return {"precision": p, "recall": r, "f1": f, "tp": tp, "fp": fp, "fn": fn}


def example_level(per_record: list[tuple[bool, bool]]) -> dict:
    """per_record: list of (gold_positive, pred_positive)."""
    tp = sum(1 for g, p in per_record if g and p)
    fp = sum(1 for g, p in per_record if (not g) and p)
    fn = sum(1 for g, p in per_record if g and (not p))
    out = prf1(tp, fp, fn)
    out["n"] = len(per_record)
    out["n_positive"] = sum(1 for g, _ in per_record if g)
    return out


def span_level(per_record_masks: list[tuple[list[bool], list[bool]]]) -> dict:
    """per_record_masks: list of (gold_mask, pred_mask) over answer characters."""
    tp = fp = fn = 0
    for gold, pred in per_record_masks:
        for g, p in zip(gold, pred):
            if p and g:
                tp += 1
            elif p and not g:
                fp += 1
            elif g and not p:
                fn += 1
    out = prf1(tp, fp, fn)
    out["n"] = len(per_record_masks)
    return out

## 4. Lexical floor baseline

A trivial baseline: predict every content word in the answer that does not appear in the context as hallucinated. This is a span-level baseline only; it is the *minimum* an actual model must beat.

In [ ]:
WORD_RE = re.compile(r"[A-Za-z0-9]+")


def words_set(text: str) -> set[str]:
    return {m.group(0).lower() for m in WORD_RE.finditer(text) if len(m.group(0)) > 2}


def lexical_predict_mask(context: str, output: str) -> list[bool]:
    ctx = words_set(context)
    mask = [False] * len(output)
    for m in WORD_RE.finditer(output):
        tok = m.group(0).lower()
        if len(tok) <= 2:
            continue
        if tok not in ctx:
            for i in range(m.start(), m.end()):
                mask[i] = True
    return mask


def lexical_predict_spans(context: str, output: str) -> list[dict]:
    """Convert the lexical mask into spans for example-level scoring."""
    mask = lexical_predict_mask(context, output)
    spans = []
    i = 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j < len(mask) and mask[j]:
                j += 1
            spans.append({"start": i, "end": j, "text": output[i:j]})
            i = j
        else:
            i += 1
    return spans

## 5. LettuceDetect inference

Wrap `HallucinationDetector(method="transformer", model_path=...)` and run it on every record. We collect:
- predicted spans `{start, end, text, confidence}` over the answer's character offsets;
- gold spans from `hallucination_labels`;
- per-record positive/negative flags for example-level scoring.

Long records that exceed 4096 tokens are skipped by LettuceDetect's tokenizer (it raises). We catch and treat as *no prediction* (predicted negative).

In [ ]:
def run_lettuce(records: list[dict], detector: HallucinationDetector) -> list[dict]:
    """Run inference and return list of dicts with gold/pred spans + masks."""
    results = []
    n_failed = 0
    for r in tqdm(records, leave=False):
        out = r["output"]
        try:
            spans = detector.predict(
                context=[r["context"]],
                question=r["query"],
                answer=out,
                output_format="spans",
            )
        except Exception:
            spans = []
            n_failed += 1
        results.append({
            "id": r.get("id"),
            "corruption_type": r["meta"]["corruption_type"],
            "output": out,
            "gold_spans": r["hallucination_labels"],
            "pred_spans": spans,
        })
    if n_failed:
        print(f"  {n_failed} records failed inference (treated as no-prediction)")
    return results

## 6. Score a result set

Given a list of per-record `{gold_spans, pred_spans, output, corruption_type}`, produce overall + per-corruption-type example-level and span-level metrics.

In [ ]:
def score_results(per_record: list[dict]) -> dict:
    by_type_ex: dict[str, list[tuple[bool, bool]]] = defaultdict(list)
    by_type_sp: dict[str, list[tuple[list[bool], list[bool]]]] = defaultdict(list)
    overall_ex: list[tuple[bool, bool]] = []
    overall_sp: list[tuple[list[bool], list[bool]]] = []

    for r in per_record:
        gold_pos = bool(r["gold_spans"])
        pred_pos = bool(r["pred_spans"])
        gold_mask = char_mask(r["output"], r["gold_spans"])
        pred_mask = char_mask(r["output"], r["pred_spans"])
        overall_ex.append((gold_pos, pred_pos))
        overall_sp.append((gold_mask, pred_mask))
        by_type_ex[r["corruption_type"]].append((gold_pos, pred_pos))
        by_type_sp[r["corruption_type"]].append((gold_mask, pred_mask))

    return {
        "example_level": {
            "overall": example_level(overall_ex),
            "by_type": {t: example_level(v) for t, v in by_type_ex.items()},
        },
        "span_level": {
            "overall": span_level(overall_sp),
            "by_type": {t: span_level(v) for t, v in by_type_sp.items()},
        },
    }

## 7. Run all baselines on all configs

For each model and each config, run inference and score. Persist raw predictions to disk so the table can be regenerated without re-running the model.

In [ ]:
all_metrics: dict[str, dict[str, dict]] = {}

print("==== Lexical floor baseline ====")
lex_metrics: dict[str, dict] = {}
for cfg, records in datasets_by_config.items():
    per_record = []
    for r in records:
        spans = lexical_predict_spans(r["context"], r["output"])
        per_record.append({
            "id": r.get("id"),
            "corruption_type": r["meta"]["corruption_type"],
            "output": r["output"],
            "gold_spans": r["hallucination_labels"],
            "pred_spans": spans,
        })
    lex_metrics[cfg] = score_results(per_record)
    ex = lex_metrics[cfg]["example_level"]["overall"]
    sp = lex_metrics[cfg]["span_level"]["overall"]
    print(f"  {cfg:14s}  example F1={ex['f1']:.3f}  span F1={sp['f1']:.3f}  (n={ex['n']})")

all_metrics["lexical"] = lex_metrics
(OUTPUT_DIR / "metrics_lexical.json").write_text(json.dumps(lex_metrics, indent=2))

In [ ]:
for model_id in MODEL_IDS:
    print(f"\n==== {model_id} ====")
    t0 = time.time()
    detector = HallucinationDetector(method="transformer", model_path=model_id)
    print(f"  loaded in {time.time() - t0:.1f}s")

    model_metrics: dict[str, dict] = {}
    short = model_id.split("/")[-1]
    raw_dir = OUTPUT_DIR / "raw" / short
    raw_dir.mkdir(parents=True, exist_ok=True)

    for cfg, records in datasets_by_config.items():
        print(f"  config={cfg}  n={len(records)}")
        t1 = time.time()
        per_record = run_lettuce(records, detector)
        elapsed = time.time() - t1
        rate = len(records) / elapsed if elapsed else 0.0
        with (raw_dir / f"{cfg}_{SPLIT}.jsonl").open("w") as f:
            for r in per_record:
                f.write(json.dumps(r) + "\n")
        m = score_results(per_record)
        model_metrics[cfg] = m
        ex = m["example_level"]["overall"]
        sp = m["span_level"]["overall"]
        print(
            f"    example  P={ex['precision']:.3f} R={ex['recall']:.3f} F1={ex['f1']:.3f}  |  "
            f"span  P={sp['precision']:.3f} R={sp['recall']:.3f} F1={sp['f1']:.3f}  |  "
            f"{rate:.1f} rec/s"
        )

    all_metrics[short] = model_metrics
    (OUTPUT_DIR / f"metrics_{short}.json").write_text(json.dumps(model_metrics, indent=2))

    del detector
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

(OUTPUT_DIR / "metrics_all.json").write_text(json.dumps(all_metrics, indent=2))

## 8. Results tables

Reproduce the **Lettuce-paper-style** layout: rows = method, columns = `{config} × {Precision, Recall, F1}` + an `overall` block.

We emit two tables, one for each granularity (mirroring Tables 2 and 3 of the paper).

In [ ]:
def build_table(all_metrics: dict, granularity: str) -> pd.DataFrame:
    """granularity in {'example_level', 'span_level'}"""
    rows = []
    for method, by_cfg in all_metrics.items():
        row = {"method": method}
        for cfg in CONFIGS:
            m = by_cfg[cfg][granularity]["overall"]
            row[(cfg, "P")] = round(m["precision"], 4)
            row[(cfg, "R")] = round(m["recall"], 4)
            row[(cfg, "F1")] = round(m["f1"], 4)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("method")
    df.columns = pd.MultiIndex.from_tuples(df.columns)
    return df


example_table = build_table(all_metrics, "example_level")
span_table = build_table(all_metrics, "span_level")

print("Example-level (Table 2 analogue):")
display(example_table)
print("\nSpan-level char-overlap (Table 3 analogue):")
display(span_table)

example_table.to_csv(OUTPUT_DIR / "example_level.csv")
span_table.to_csv(OUTPUT_DIR / "span_level.csv")

## 9. Per-corruption-type breakdown (combined config)

The `combined` config contains `clean`, `contradiction`, `missing_tool`, and `overgeneration` records. Breaking the metrics down by `meta.corruption_type` shows which hallucination kinds each baseline catches and which it misses.

For *clean* records, span-level recall/F1 is undefined (no gold positives), so only example-level numbers are meaningful (precision ≡ 1 - false-positive-rate).


In [ ]:
def build_by_type_table(all_metrics: dict, granularity: str, cfg: str = "combined") -> pd.DataFrame:
    rows = []
    for method, by_cfg in all_metrics.items():
        by_type = by_cfg[cfg][granularity]["by_type"]
        for ctype, m in by_type.items():
            rows.append({
                "method": method,
                "corruption_type": ctype,
                "n": m["n"],
                "precision": round(m["precision"], 4),
                "recall": round(m["recall"], 4),
                "f1": round(m["f1"], 4),
            })
    return pd.DataFrame(rows).pivot_table(
        index="method",
        columns="corruption_type",
        values=["precision", "recall", "f1"],
        aggfunc="first",
    )


print("Per-type, example-level (combined config):")
display(build_by_type_table(all_metrics, "example_level", "combined"))
print("\nPer-type, span-level (combined config):")
display(build_by_type_table(all_metrics, "span_level", "combined"))

## 10. Qualitative inspection

Show a handful of records from `combined/test` with their gold spans and the predictions of the *large* LettuceDetect model. Useful for spotting failure modes (missed `missing_tool` insertions, partial `contradiction` recall, false positives on JSON literals).

In [ ]:
import html
from IPython.display import HTML


def highlight(output: str, gold: list[dict], pred: list[dict]) -> str:
    g = char_mask(output, gold)
    p = char_mask(output, pred)
    parts = []
    for i, ch in enumerate(output):
        gg, pp = g[i], p[i]
        ch = html.escape(ch)
        if gg and pp:
            parts.append(f"<span style='background:#9f9'>{ch}</span>")  # TP
        elif gg:
            parts.append(f"<span style='background:#fdd;text-decoration:underline'>{ch}</span>")  # FN
        elif pp:
            parts.append(f"<span style='background:#ffd'>{ch}</span>")  # FP
        else:
            parts.append(ch)
    return "".join(parts)


large_short = MODEL_IDS[-1].split("/")[-1]
raw_path = OUTPUT_DIR / "raw" / large_short / f"combined_{SPLIT}.jsonl"
if raw_path.exists():
    sample_records = [json.loads(l) for l in raw_path.read_text().splitlines()]
    examples = [r for r in sample_records if r["corruption_type"] != "clean"][:5]
    blocks = [
        "<style>body{font-family:system-ui}</style>"
        "<p><b>Legend:</b> "
        "<span style='background:#9f9'>TP</span> "
        "<span style='background:#fdd;text-decoration:underline'>FN (gold only)</span> "
        "<span style='background:#ffd'>FP (pred only)</span></p>"
    ]
    for r in examples:
        blocks.append(f"<hr><b>{r['id']}</b> — corruption_type=<code>{r['corruption_type']}</code>")
        blocks.append(f"<div style='white-space:pre-wrap'>{highlight(r['output'], r['gold_spans'], r['pred_spans'])}</div>")
    display(HTML("\n".join(blocks)))
else:
    print(f"raw predictions not found at {raw_path}; rerun the inference cell first.")

## 11. Discussion

What this notebook delivers, against the task spec from `Hallucination Detection in Tool Calling.pdf` (§2: *Evaluate existing baseline methods*):

- **Method**: LettuceDetect (§ 4 of the Lettuce paper) — ModernBERT-based token classifier `c, q, a` → per-token `{supported, hallucinated}`, span-collapsed at threshold 0.5.
- **Mapping** (§ 2 of the task PDF): `query → question`, `tool output → context`, `assistant final response → answer`.
- **Datasets** (§ 1 of the task PDF): the four published configs of `Ivan1008/toolace-hallucination-spans`, one per corruption type plus `combined`. Each carries clean negatives so example-level precision is meaningful.
- **Reporting** (§ 5 of the Lettuce paper): two tables — example-level (binary) and span-level (character-overlap micro-F1).
- **Sanity baseline**: lexical "word-not-in-context" floor.

**Expected qualitative pattern** (also matches what the dataset card reports for validation):
- Span-level F1 is low across the board because LettuceDetect was trained on RAGTruth (news / QA / data-to-text) where the *context* is prose, not JSON-serialized tool output.
- `missing_tool` is hardest: the corrupted span is a *plausible* assistant offer ("would you like me to book the flight?") that is consistent with the context but off-policy. Without seeing the available-tools list, an encoder can't catch it.
- `overgeneration` is the easiest: inserted sentences contain content words absent from the (short, structured) tool output, and even the lexical baseline catches some.
- `contradiction` is the hardest from a precision standpoint: replaced tokens are typically present in the global content-word distribution, so LettuceDetect's per-token classifier flags many false positives.

These results define the **baseline floor** that step § 3 (*Improve the baseline approaches*) of the task must beat — e.g. by fine-tuning LettuceDetect on the train split of this dataset (a single-line change to the existing trainer) or by feeding the available-tools schema into the context, which would lift `missing_tool` recall in particular.